### HW5 - Neural Collaborative Filtering (GMF)

For this programming assignment, you will implement the Generalized Matrix Factorization model using neural networks.
    
A skeleton code is provided and you need to fill in the remaining parts and label (via comments) some other parts of the code.

You will run this using our MovieNight dataset. The source code and dataset are linked in the supplemental HW5 assignment page on Canvas.

#### Submission:
Update and run your code. Submit the .pdf of your file in Gradescope.

#### Deadlines:
Late submission (one day late) incurs a 50% penalty.
    

![image.png](attachment:image.png)

In [ ]:
%pip install tensorflow

In [2]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import keras
from keras import layers
import tensorflow as tf
from keras.models import Model
from keras.optimizers import Adam

In [5]:
#grab data and view
rating_df = pd.read_csv('movie_night_utilitymatrix_S26.csv')
rating_df = rating_df.rename(columns={'user':'u_id','movie':'m_id','rating':'rate'})
rating_df.head()

,u_id,m_id,rate
0,1,2,5.0
1,1,3,5.0
2,1,7,3.0
3,1,11,2.0
4,1,14,5.0


In [6]:
# Model may require seed for embeddings to work depending on keras/tf version
# rating_df = rating_df.sample(frac=1, random_state=73)

#splits data into inputs (X) and ratings (y)
X = rating_df[['u_id', 'm_id']].values
y = rating_df["rate"]

In [7]:
#create the train and test set. test is 25%
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    stratify=y,
                                                    test_size=0.25)

print('> Train set ratings: {}'.format(len(y_train)))
print('> Test set ratings: {}'.format(len(y_test)))

> Train set ratings: 4144
> Test set ratings: 1382


In [8]:
#formating input for embedding
X_train_array = [X_train[:, 0], X_train[:, 1]]
X_test_array = [X_test[:, 0], X_test[:, 1]]

In [27]:
# the input dimension is the number of unique entries
in_u_dim = rating_df["u_id"].max() + 1
in_m_dim = rating_df["m_id"].max() + 1

latent_out_dim = 8 #desired embedding length, determines complexity

### Instructions!
Take a look at the General Matrix Factorization Architecture.

#### Task 1:
Replace each ###label me### comment with the correct GMF architectural component: single neuron, element-wise multiplication layer, or input layer.

#### Task 2:
Add an embedding layer for the users at the point in code denoted with: ###User Embedding should go here ###

#### Task 3:
Additionally, fill out code wherever you see ##
please check the #comments for instructions



In [28]:
# Model
from keras.layers import Add, Activation, Lambda, BatchNormalization, Concatenate, Dropout, Input, Embedding, Dot, Reshape, Dense, Flatten

def GMFact():


    # Input Layer
    user = Input(name = 'u_in', shape = [1])
    movie = Input(name = 'm_in', shape = [1])

    movie_embedding = Embedding(name = 's_emb',
                       input_dim = in_m_dim,
                       output_dim = latent_out_dim)(movie)


    ### User Embedding should go here ###
    user_embedding = Embedding(name = 'u_emb',
                       input_dim = in_u_dim,
                       output_dim = latent_out_dim)(user)


    # Element-wise Multiply Layer
    x = tf.keras.layers.Multiply()([user_embedding, movie_embedding])      #We need to specify the correct layer type to element-wise multiply the user_embedding with the movie_embedding.
                                                                    #Remember, you still need a vector, not a single digit.
                                                                    #Hint: the name of the layer is one of these: https://keras.io/api/layers/merging_layers/
    x = Flatten()(x)



    #Single Neuron
    x = Dense(1, kernel_initializer='lecun_uniform')(x)                        #add a single neuron and set the 'kernel_initializer' to 'lecun_uniform' see example: https://github.com/fractus-io/keras-tutorial/blob/master/README.md#initializers
    x = Activation("sigmoid")(x)



    model = Model(inputs=[user, movie], outputs=x)

    model.compile(
      optimizer='sgd',
      loss='mse',
      metrics=[tf.keras.metrics.RootMeanSquaredError()])

    return model

model = GMFact()
model.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ u_in (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ m_in (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ u_emb (Embedding)   │ (None, 1, 8)      │      4,664 │ u_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s_emb (Embedding)   │ (None, 1, 8)      │        168 │ m_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_5          │ (None, 1, 8)      │          0 │ u_emb[0][0],      │
│ (Multiply)          │                   │            │ s_emb[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 8)         │          0 │ multiply_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 1)         │          9 │ flatten_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 1)         │          0 │ dense_6[0][0]     │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,841 (18.91 KB)

 Trainable params: 4,841 (18.91 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [33]:
batch_size = 50
history = model.fit(
    x=X_train_array,
    y=y_train,
    batch_size=batch_size,
    epochs=1,
    verbose=1,
    validation_data=(X_test_array, y_test)
)

model.save_weights('gmf.weights.h5')

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.0106 - root_mean_squared_error: 3.1640 - val_loss: 9.8889 - val_root_mean_squared_error: 3.1447


In [34]:
final_RMSE = (history.history['root_mean_squared_error'])[-1]
final_RMSE

3.1639604568481445